# Статистический анализ точности системы лазерного наведения с БПЛА

В этой главе оценивается точность системы лазерного наведения, установленной на квадрокоптере со стабилизированным подвесом. Практический вопрос состоит в том, при каких условиях полёта лазерное пятно остаётся достаточно близко к заданной реперной точке.

Поскольку отклонение пятна имеет две координатные составляющие, для анализа нужен один итоговый показатель ошибки. Таким показателем является радиальная ошибка

$$
R = \sqrt{r_x^2 + r_y^2}.
$$

Здесь $r_x$ и $r_y$ — отклонения пятна по двум горизонтальным осям, мм. Иными словами, $R$ показывает не ошибку по отдельной оси, а полное расстояние от фактического положения пятна до заданной точки. Поэтому далее именно $R$ используется как результативный признак.

### Характеристика выборки

| Параметр | Значение |
|---|---|
| Период проведения испытаний | апрель — октябрь 2025 г. |
| Число лётных дней | 59 |
| Контрольные точки | 5 (P1–P5) |
| Высоты зависания | $H \in \{3,\; 5,\; 10,\; 15,\; 20\}$ м |
| Повторов на каждую комбинацию (точка × высота) | 3 |
| Общее число измерений | $59 \times 5 \times 5 \times 3 = 4425$ |

Для каждого измерения фиксировались высота $H$, локальная скорость ветра $V$, число спутников GNSS, метеоусловия и координатные ошибки $r_x$, $r_y$. Прежде чем анализировать влияние этих факторов на $R$, проверим, что выборка охватывает разные погодные режимы и не сводится к одному типу условий.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


def show_eq(model, response="R", as_log=False):
    b = model.params
    lhs = r"\widehat{\ln R}" if as_log else rf"\hat{{{response}}}"
    parts = [f"{b['Intercept']:.3g}"]
    for name in ["H", "V", "sat", "H:V"]:
        if name in b:
            sym = {"H": "H", "V": "V", "sat": r"\mathrm{sat}", "H:V": r"H\!\cdot\!V"}[name]
            parts.append(f"{b[name]:+.3g}\\,{sym}")
    eq = " ".join(parts).replace("+ -", "- ")
    display(Markdown(rf"$${lhs} = {eq} \quad (R^2 = {model.rsquared:.3f})$$"))


BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
JSON_PATH = BASE_DIR / "balanced_selection" / "selected_days_2024-2025.json"

H_ORDER = [3, 5, 10, 15, 20]
H_CAT_ORDER = ["3 м", "5 м", "10 м", "15 м", "20 м"]
ALPHA = 0.05

COL_MAP = {
    "Время": "time", "Точка": "point", "H (м)": "H", "Полёт": "flight",
    "V (м/с)": "V", "Порывы, откр. источник (м/с)": "gusts",
    "Направление ветра": "wind_dir", "Облачность (%)": "cloud",
    "Спутники": "sat", "r_x (мм)": "r_x", "r_y (мм)": "r_y", "R (мм)": "R"
}


def load_measurements(input_dir: Path) -> pd.DataFrame:
    frames = []
    for fpath in sorted(input_dir.glob("*.xlsx")):
        if fpath.name.startswith("~$"):
            continue
        tmp = pd.read_excel(fpath, header=12, engine="openpyxl")
        tmp.rename(columns=COL_MAP, inplace=True)
        tmp["date"] = fpath.stem
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    for c in ("H", "V", "cloud", "sat", "R", "r_x", "r_y", "gusts", "wind_dir"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.dropna(subset=["H", "V", "R"], inplace=True)
    return df


weather = pd.DataFrame(json.loads(JSON_PATH.read_text(encoding="utf-8")))
weather["date"] = pd.to_datetime(weather["date"])

df = load_measurements(OUTPUT_DIR)
df["H_cat"] = df["H"].astype(int).astype(str) + " м"
df["V_bin"] = pd.cut(
    df["V"],
    bins=[0, 2.2, 2.9, 3.6, 4.4, 100],
    labels=["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"],
)

print(f"Загружено: {len(df)} измерений за {df['date'].nunique()} дней")
print(f"Погодный JSON: {len(weather)} дней")
print(f"Высоты: {sorted(df['H'].unique())} м")
print(f"Точки: {', '.join(sorted(df['point'].unique()))}")
print(f"Повторов на высоту: {df['flight'].nunique()}")

ALL_FIGS: list[tuple[str, go.Figure]] = []


Загружено: 4425 измерений за 59 дней
Погодный JSON: 59 дней
Высоты: [np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20)] м
Точки: P1, P2, P3, P4, P5
Повторов на высоту: 3


### Обзор выборки и условий эксперимента

Разнообразие погодных условий важно проверить до анализа ошибки: если все полёты выполнены в одном режиме, влияние высоты, ветра и спутников может быть смешано с особенностями конкретных дней. Поэтому сначала рассмотрим, какие типы погоды вошли в выборку и в каких диапазонах изменялись основные метеопараметры.

In [2]:
weather_counts = weather["weather_code_text"].value_counts().reset_index()
weather_counts.columns = ["Погода", "Дней"]

fig = px.bar(
    weather_counts, x="Погода", y="Дней",
    color="Погода", text="Дней",
    title="Рис. 1. Распределение лётных дней по типу погоды",
)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Количество дней")
fig.update_traces(textposition="outside")
ALL_FIGS.append(("fig_01_weather_types", fig))
fig.show()

In [3]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Температура (°C)", "Средний ветер (м/с)",
    "Максимальные порывы (м/с)", "Облачность (%)"
])

for i, (col, name) in enumerate([
    ("temperature_2m_mean", "Температура"),
    ("wind_speed_10m_mean", "Ветер"),
    ("wind_gusts_10m_max", "Порывы"),
    ("cloud_cover_mean", "Облачность"),
]):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=weather[col], nbinsx=15, name=name, showlegend=False),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=500, title_text="Рис. 2. Распределение метеопараметров по дням выборки")
ALL_FIGS.append(("fig_02_meteo_histograms", fig))
fig.show()

In [4]:
fig = px.histogram(
    df, x="R", nbins=80, marginal="box",
    title="Рис. 3. Общее распределение радиальной ошибки R",
    labels={"R": "R (мм)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(yaxis_title="Количество измерений", height=400)
ALL_FIGS.append(("fig_03_R_distribution", fig))
fig.show()

Рис. 1 показывает, что выборка не ограничена одним погодным сценарием: помимо 27 пасмурных дней, в неё вошли дни с моросью разной интенсивности, ясная погода и переменная облачность. Такое распределение важно для дальнейшего анализа, потому что ошибка наведения будет оцениваться не только в спокойных, но и в неблагоприятных условиях.

На рис. 2 метеопараметры покрывают широкий рабочий диапазон: средний ветер меняется примерно от 1 до 7 м/с, максимальные порывы — примерно от 4 до 18 м/с, облачность — от почти нулевой до 100%. Следовательно, в выборке присутствуют как мягкие, так и напряжённые режимы полёта.

Общее распределение радиальной ошибки на рис. 3 правоасимметрично: основная масса измерений сосредоточена примерно в диапазоне 50–250 мм, но правый хвост уходит за 1000 мм. Иными словами, типичная ошибка умеренная, однако при неблагоприятных сочетаниях условий появляются крупные промахи. Этот факт важно сохранить: позже он объяснит, почему одной линейной модели на исходной шкале недостаточно и потребуется рассмотреть логарифмическую форму.

### Описательная статистика

После общего обзора выборки нужно перейти от отдельных измерений к компактным характеристикам ошибки. Для каждой высоты сначала вычисляется среднее значение радиальной ошибки

$$
\bar{R} = \frac{1}{n}\sum_{i=1}^{n} R_i.
$$

Здесь $R_i$ — ошибка в отдельном измерении, $n$ — число измерений в высотной группе. Иными словами, $\bar{R}$ показывает типичный уровень ошибки на данной высоте.

Одного среднего недостаточно: две группы могут иметь близкие средние, но разную устойчивость результата. Поэтому дополнительно рассчитывается стандартное отклонение

$$
\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(R_i - \bar{R})^2}.
$$

Эта величина показывает, насколько отдельные измерения отклоняются от среднего уровня. Если вместе со средним растёт и $\sigma$, то ухудшается не только точность, но и повторяемость наведения.

In [5]:
desc = (
    df.groupby("H", sort=True)["R"]
    .agg(
        N="count",
        Среднее="mean",
        Ст_откл="std",
        Минимум="min",
        Максимум="max",
        P95=lambda x: x.quantile(0.95)
    )
    .round(1)
)
desc.index.name = "H (м)"
desc.columns = ["N", "Среднее, мм", "σ, мм", "Мин, мм", "Макс, мм", "R₉₅, мм"]

fig = go.Figure(data=[go.Table(
    columnwidth=[50, 50, 80, 70, 60, 70, 70],
    header=dict(
        values=["<b>H (м)</b>"] + [f"<b>{c}</b>" for c in desc.columns],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=[desc.index.tolist()] + [desc[c].tolist() for c in desc.columns],
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 1. Описательная статистика радиальной ошибки R по высотам",
    width=820, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_desc_table", fig))
fig.show()

n_per_h = len(df) // df["H"].nunique()
n_days = df["date"].nunique()
print(f"{n_per_h} измерений на высоту: {n_days} дней × 5 точек × 3 повтора = {n_per_h}")

print("\nОписательная статистика непрерывных ковариат:")
for col, unit in [("V", "м/с"), ("gusts", "м/с"), ("cloud", "%"), ("sat", "шт.")]:
    vals = df[col].dropna()
    print(f"  {col}: среднее = {vals.mean():.1f} {unit}, σ = {vals.std():.1f}, "
          f"диапазон [{vals.min():.1f} – {vals.max():.1f}]")

885 измерений на высоту: 59 дней × 5 точек × 3 повтора = 885

Описательная статистика непрерывных ковариат:
  V: среднее = 3.4 м/с, σ = 1.4, диапазон [0.7 – 8.7]
  gusts: среднее = 9.9 м/с, σ = 3.1, диапазон [3.1 – 19.2]
  cloud: среднее = 60.1 %, σ = 33.9, диапазон [0.0 – 100.0]
  sat: среднее = 22.4 шт., σ = 3.7, диапазон [16.0 – 31.0]


Таблица показывает, что средний уровень ошибки растёт монотонно: от $68{,}1$ мм на высоте 3 м до $303{,}1$ мм на высоте 20 м. Рост почти в 4,5 раза означает, что высота уже на уровне описательной статистики выступает основным кандидатом на главный фактор ошибки.

Одновременно увеличивается стандартное отклонение: с $23{,}8$ мм на 3 м до $142{,}5$ мм на 20 м. Иными словами, на больших высотах система не только промахивается сильнее в среднем, но и даёт менее устойчивые повторные измерения.

Важно, что разброс растёт вместе с уровнем ошибки. Это указывает на пропорциональную, или мультипликативную, природу процесса: крупная ошибка сопровождается крупной изменчивостью. Далее проверим тот же вывод не только по таблице, но и по форме распределений в высотных группах.

### Распределение ошибки по высотам

Среднее и стандартное отклонение показывают общий уровень ошибки, но не раскрывают форму распределения. Поэтому далее рассмотрим, как с высотой меняются медиана, межквартильный размах и правый хвост распределения $R$.

In [6]:
fig = px.box(
    df, x="H", y="R", color="H_cat",
    title="Рис. 4. Распределение радиальной ошибки R по высоте полёта",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_layout(showlegend=False, height=500, width=850)
ALL_FIGS.append(("fig_04_boxplot_H", fig))
fig.show()

In [7]:
fig = px.violin(
    df, x="H", y="R", color="H_cat", box=True, points=False,
    title="Рис. 5. Форма распределения R по высотам (скрипичная диаграмма)",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_traces(width=0.85, spanmode="hard")
fig.update_layout(showlegend=False, height=600, width=900, violingap=0.15)
ALL_FIGS.append(("fig_05_violin_H", fig))
fig.show()

In [8]:
stats_by_h = df.groupby("H")["R"].agg(["mean", "std", "count"]).reset_index()
stats_by_h["se"] = stats_by_h["std"] / np.sqrt(stats_by_h["count"])
stats_by_h["ci95"] = stats_by_h["se"] * 1.96

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=stats_by_h["H"], y=stats_by_h["mean"],
    mode="lines+markers", name="Среднее R",
    line=dict(width=3), marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=list(stats_by_h["H"]) + list(stats_by_h["H"][::-1]),
    y=list(stats_by_h["mean"] + stats_by_h["ci95"])
      + list((stats_by_h["mean"] - stats_by_h["ci95"])[::-1]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)",
    line=dict(width=0), name="95% дов. интервал", showlegend=True,
))
fig.update_layout(
    title="Рис. 6. Средняя ошибка R по высоте (с 95% доверительным интервалом)",
    xaxis_title="Высота H (м)", yaxis_title="R (мм)", height=400,
)
ALL_FIGS.append(("fig_06_mean_R_ci", fig))
fig.show()

Рис. 4 и 5 показывают не только рост центрального уровня ошибки, но и изменение всей формы распределения. Медиана $R$ смещается примерно от $65$ мм на 3 м до $264$ мм на 20 м, а межквартильный размах расширяется от 50–83 мм до 208–358 мм. Значит, на больших высотах увеличивается не только типичная ошибка, но и диапазон обычных, неэкстремальных значений.

Правый хвост распределения также становится длиннее: на высоте 20 м отдельные значения доходят примерно до 1000 мм. Такие точки не следует воспринимать как единственную причину роста среднего; основной сдвиг виден уже по медиане и межквартильному размаху.

На рис. 6 средние значения возрастают монотонно, а 95% доверительные интервалы между высотами практически не перекрываются. Следовательно, различия между высотными группами нельзя объяснить только случайным разбросом повторных измерений.

Физически этот результат связан с геометрическим рычагом. При малой угловой нестабильности подвеса $\delta\varphi$ линейное смещение пятна можно приближённо записать как

$$
\Delta R \approx H \cdot \tan(\delta\varphi) \approx H \cdot \delta\varphi.
$$

Иными словами, одна и та же угловая ошибка на большей высоте даёт больший промах на поверхности. Дальше проверим, почему прирост особенно усиливается на участке 15–20 м.

### Влияние рельефа на высотах 15–20 м

Монотонный рост ошибки с высотой объясняется геометрическим рычагом, но участок 15–20 м требует отдельной проверки. Испытания проводились на территории карьера: до 10–15 м БПЛА находится внутри карьерной выемки, где рельеф частично экранирует ветровое воздействие, а при подъёме выше бровки попадает в более открытый поток.

Отсюда возникает проверяемая гипотеза: если рельеф действительно ослабляет ветер на меньших высотах, то при переходе к 20 м ошибка должна расти быстрее, а различие между слабыми и сильными ветровыми режимами должно увеличиваться. Проверим это по медианным ошибкам и по приростам между соседними высотами.

In [9]:
v_q25 = df["V"].quantile(0.25)
v_q75 = df["V"].quantile(0.75)

low_wind = df[df["V"] <= v_q25].groupby("H")["R"].median().reset_index()
low_wind.columns = ["H", "R_median"]
low_wind["Режим"] = f"Слабый ветер (V ≤ {v_q25:.1f} м/с)"

high_wind = df[df["V"] >= v_q75].groupby("H")["R"].median().reset_index()
high_wind.columns = ["H", "R_median"]
high_wind["Режим"] = f"Сильный ветер (V ≥ {v_q75:.1f} м/с)"

plateau_df = pd.concat([low_wind, high_wind])

fig = px.line(
    plateau_df, x="H", y="R_median", color="Режим",
    markers=True,
    title="Рис. 7. Медианная ошибка R по высоте: слабый и сильный ветер",
    labels={"H": "Высота H (м)", "R_median": "Медианная R (мм)"},
)
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(height=450)
ALL_FIGS.append(("fig_07_plateau_wind", fig))
fig.show()

In [10]:
heights = sorted(df["H"].unique())
median_by_h = df.groupby("H")["R"].median()

deltas = []
for i in range(1, len(heights)):
    h_prev, h_curr = heights[i - 1], heights[i]
    delta = median_by_h[h_curr] - median_by_h[h_prev]
    deltas.append({"Интервал высот": f"{int(h_prev)}→{int(h_curr)} м", "ΔR": delta})

delta_df = pd.DataFrame(deltas)

fig = px.bar(
    delta_df, x="Интервал высот", y="ΔR", text="ΔR",
    title="Рис. 8. Прирост медианной ошибки между соседними высотами",
    labels={"Интервал высот": "Интервал высот (переход)", "ΔR": "ΔR (мм)"},
    color_discrete_sequence=["coral"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=400, yaxis_title="Прирост ΔR (мм)", xaxis_title="Интервал высот (переход)")
ALL_FIGS.append(("fig_08_delta_R", fig))
fig.show()

Рис. 7 и 8 согласуются с гипотезой о выходе БПЛА из ветровой тени карьера. Прирост медианной ошибки между соседними высотами составляет $15{,}8$ мм при переходе 3→5 м, $44{,}3$ мм при 5→10 м, $49{,}3$ мм при 10→15 м и уже $89{,}3$ мм при 15→20 м. Последний переход даёт наибольший прирост, то есть рост ошибки не замедляется, а ускоряется.

На рис. 7 видно, что линии слабого и сильного ветра расходятся с высотой. На малых высотах различие между режимами умеренное, а к 20 м разрыв становится максимальным: сильный ветер поднимает медианную ошибку примерно до 440 мм, тогда как при слабом ветре она остаётся около 200 мм.

Иными словами, на малых высотах ошибка в основном задаётся рычагом $H$, а выше бровки карьера к нему добавляется ветровая составляющая. Этот результат подводит к следующему шагу: нужно отдельно рассмотреть скорость ветра $V$ и затем проверить в регрессии член взаимодействия $H \times V$.

### Влияние ветра

После высоты следующим фактором является скорость ветра $V$. Ветер увеличивает угловую нестабильность подвеса: чем больше $V$, тем больше амплитуда малых отклонений $\delta\varphi(V)$.

Если подставить это в геометрическую связь

$$
\Delta R \approx H \cdot \delta\varphi(V),
$$

получается важный вывод: влияние ветра должно усиливаться с высотой. Поэтому далее нас интересует не только отдельная связь $R$ с $V$, но и будущий член взаимодействия $H \times V$ в регрессионной модели.

In [11]:
fig = px.scatter(
    df, x="V", y="R", color="H_cat",
    title="Рис. 9. Зависимость R от скорости ветра V (цвет — высота)",
    labels={"V": "Скорость ветра V (м/с)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35, trendline="ols",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
ALL_FIGS.append(("fig_09_R_vs_V", fig))
fig.show()

In [12]:
wind_bins = ["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"]

median_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="median", observed=False
).reindex(columns=wind_bins)
mean_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="mean", observed=False
).reindex(columns=wind_bins)

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Heatmap(
        z=median_pivot.to_numpy(),
        x=median_pivot.columns.tolist(),
        y=median_pivot.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="R (мм)", x=1.02, y=0.82, len=0.34),
        text=np.round(median_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Медиана R: %{z:.1f} мм<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=mean_pivot.to_numpy(),
        x=mean_pivot.columns.tolist(),
        y=mean_pivot.index.tolist(),
        colorscale="Blues",
        colorbar=dict(title="R (мм)", x=1.02, y=0.18, len=0.34),
        text=np.round(mean_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Среднее R: %{z:.1f} мм<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="Рис. 10. Ошибка R (мм): медиана и среднее по высоте и режиму ветра",
    height=800,
)
fig.update_xaxes(side="top", row=1, col=1)
fig.update_xaxes(side="top", row=2, col=1)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=2,
    col=1,
)
ALL_FIGS.append(("fig_10_heatmap_HV", fig))
fig.show()

На рис. 9 все линии тренда имеют положительный наклон: при увеличении скорости ветра $V$ радиальная ошибка $R$ возрастает на каждой высоте. При этом наклон линии для 20 м заметно больше, чем для 3–5 м, то есть один и тот же прирост ветра даёт более крупный промах на большей высоте.

Рис. 10 показывает тот же результат в агрегированном виде. В зоне 3 м и слабого ветра медианная ошибка составляет около 45 мм, а в зоне 20 м и сильного ветра — около 463 мм. Рост более чем в десять раз возникает не из-за одного фактора, а из-за их совместного действия.

По вертикали тепловой карты изменение сильнее, чем по горизонтали: переход к большей высоте увеличивает ошибку заметнее, чем переход к соседнему ветровому режиму. Поэтому графики подтверждают рабочую иерархию $H > V$, но одновременно показывают, что член $H \times V$ должен быть проверен регрессионно.

### Вторичные факторы: направление ветра, спутники, облачность

Высота и скорость ветра уже дают основную физическую картину, но в журнале также фиксировались направление ветра, число видимых спутников GNSS и облачность. Эти признаки нужно проверить отдельно: часть из них может иметь собственное влияние, а часть — лишь сопровождать уже найденные факторы.

In [13]:
sample_rxy = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample_rxy, x="r_x", y="r_y", color="wind_dir",
    color_continuous_scale="HSV",
    title="Рис. 11. Компоненты ошибки r_x и r_y (цвет — направление ветра)",
    labels={"r_x": "r_x (мм)", "r_y": "r_y (мм)", "wind_dir": "Направл. (°)"},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=4))

r_max = max(sample_rxy["r_x"].abs().max(), sample_rxy["r_y"].abs().max()) * 1.05
radii = np.linspace(r_max * 0.25, r_max, 4)
theta_circle = np.linspace(0, 2 * np.pi, 361)

for r in radii:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta_circle), y=r * np.sin(theta_circle),
        mode="lines", line=dict(color="gray", width=0.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

compass = {"С": 90, "СВ": 45, "В": 0, "ЮВ": 315, "Ю": 270, "ЮЗ": 225, "З": 180, "СЗ": 135}
for label, deg in compass.items():
    rad = np.radians(deg)
    fig.add_trace(go.Scatter(
        x=[0, r_max * np.cos(rad)], y=[0, r_max * np.sin(rad)],
        mode="lines", line=dict(color="lightgray", width=0.5),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_annotation(
        x=r_max * 1.08 * np.cos(rad), y=r_max * 1.08 * np.sin(rad),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )

fig.update_layout(
    height=620, width=650,
    xaxis=dict(scaleanchor="y", range=[-r_max * 1.15, r_max * 1.15]),
    yaxis=dict(range=[-r_max * 1.15, r_max * 1.15]),
)
ALL_FIGS.append(("fig_11_rxy_winddir", fig))
fig.show()

На рис. 11 облако точек $(r_x, r_y)$ в центральной области близко к радиально симметричному: ошибки распределены вокруг начала координат без устойчивого вытягивания в одном направлении. Цвет, соответствующий азимуту ветра, перемешан по квадрантам, поэтому выраженной направленной связи между вектором ошибки и направлением ветра не наблюдается.

Для дальнейшей модели это означает, что направление ветра не требуется вводить как отдельную векторную пару признаков. Достаточно использовать скалярную скорость $V$, которая отражает силу ветрового воздействия на подвес.

In [14]:
df["sat_group"] = pd.qcut(df["sat"], 4, labels=False, duplicates="drop")
sat_labels = (
    df.groupby("sat_group")["sat"]
    .agg(["min", "max"])
    .apply(lambda r: f"{int(r['min'])}–{int(r['max'])}", axis=1)
)
df["sat_label"] = df["sat_group"].map(sat_labels)
sat_order = sorted(sat_labels.values, key=lambda s: int(s.split("–")[0]))

fig = px.box(
    df, x="sat_label", y="R",
    title="Рис. 12. Ошибка R по квартилям числа спутников",
    labels={"sat_label": "Спутники (квартили)", "R": "R (мм)"},
    color_discrete_sequence=["mediumpurple"],
    category_orders={"sat_label": sat_order},
)
fig.update_layout(height=400)
ALL_FIGS.append(("fig_12_sat_boxplot", fig))
fig.show()

Рис. 12 показывает слабый, но устойчивый спад ошибки при увеличении числа видимых спутников GNSS. В нижнем квартиле спутников (16–20) медиана $R$ составляет около 153 мм, а в верхнем квартиле (26–31) — около 97 мм.

Этот результат имеет понятный физический смысл: большее число спутников улучшает позиционную оценку БПЛА и снижает ошибку удержания. Поэтому в регрессионной модели ожидается отрицательный коэффициент при числе спутников: при прочих равных увеличение `sat` должно уменьшать $R$.

In [15]:
factors = ["H", "V", "sat", "cloud"]
factor_names = {"H": "Высота", "V": "Ветер", "sat": "Спутники", "cloud": "Обл."}
corrs = []
for f in factors:
    rho, p = stats.spearmanr(df[f], df["R"])
    corrs.append({
        "Фактор": factor_names[f],
        "ρ Спирмена": rho,
        "p-значение": p,
        "|ρ|": abs(rho)
    })

corr_df = pd.DataFrame(corrs)

fig = px.bar(
    corr_df, x="Фактор", y="ρ Спирмена", text="ρ Спирмена",
    color="|ρ|", color_continuous_scale="Blues",
    title="Рис. 13. Ранговая корреляция Спирмена каждого фактора с R",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ρ Спирмена")
ALL_FIGS.append(("fig_13_spearman", fig))
fig.show()

Рис. 13 сводит предварительный графический анализ в одну корреляционную картину. Самая сильная монотонная связь с ошибкой наблюдается у высоты: $\rho_H = 0{,}825$. Ветер занимает второе место: $\rho_V = 0{,}401$. Число спутников имеет отрицательную связь с ошибкой, $\rho_{\text{sat}} = -0{,}246$, что согласуется с предыдущим графиком.

Облачность даёт положительную корреляцию $\rho_{\text{cloud}} = 0{,}249$, но этот результат нельзя читать как самостоятельное физическое влияние облаков на лазерное наведение. В рассматриваемой выборке пасмурные дни чаще сопровождаются более сильным ветром, поэтому облачность может выступать косвенным маркером неблагоприятной погоды.

Итак, графический и корреляционный анализ задают предварительную иерархию факторов: высота — основной источник изменения $R$, ветер — второй, число спутников — слабый корректирующий фактор, облачность требует проверки на самостоятельный вклад. Формально подтвердим эту структуру с помощью дисперсионного анализа.


### Проверка предпосылок дисперсионного анализа

Графики уже показали, что ошибка растёт с высотой и ветром, но перед дисперсионным анализом нужно проверить, насколько данные согласуются с его базовыми предпосылками. Здесь важны две проверки: форма распределения внутри высотных групп и однородность дисперсий между группами.

Критерий Шапиро–Уилка проверяет нулевую гипотезу о нормальности распределения. Его статистика $W$ лежит в интервале от 0 до 1: чем ближе $W$ к единице, тем ближе выборка к нормальному закону. При этом $p$-значение нужно читать аккуратно: на большой выборке даже умеренные отклонения становятся формально значимыми.

In [16]:
H_ORDER = sorted(df["H"].unique())
groups_by_H = [df.loc[df["H"] == h, "R"].values for h in H_ORDER]

shapiro_rows = []
for h, grp in zip(H_ORDER, groups_by_H):
    sample = (
        grp if len(grp) <= 5000
        else np.random.default_rng(42).choice(grp, 5000, replace=False)
    )
    W, p = stats.shapiro(sample)
    shapiro_rows.append([
        int(h), len(grp), round(W, 4), f"{p:.2e}",
        "Норм." if p > ALPHA else f"W={W:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    columnwidth=[60, 60, 80, 100, 120],
    header=dict(
        values=["<b>H (м)</b>", "<b>N</b>", "<b>W</b>", "<b>p-значение</b>", "<b>Заключение</b>"],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=list(zip(*shapiro_rows)),
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 2. Результаты критерия Шапиро–Уилка по высотным группам",
    width=700, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_shapiro_table", fig))
fig.show()

В каждой высотной группе содержится $N = 885$ измерений, поэтому критерий Шапиро–Уилка обладает высокой мощностью. Значения $W$ снижаются от $0{,}976$ на 3 м до $0{,}872$ на 20 м: распределения остаются близкими к нормальному ядру, но на больших высотах усиливается правая асимметрия и влияние хвостов.

Формально $p$-значения малы во всех группах, однако это не означает непригодность дальнейшего анализа. При таком объёме выборки важнее учитывать характер отклонения: оно ожидаемо для величины $R \geq 0$ и связано с редкими крупными промахами. Следующая предпосылка — равенство дисперсий между высотными группами.

#### Однородность дисперсий. Критерий Ливеня

Критерий Ливеня проверяет гипотезу

$$
H_0\colon \sigma^2_1 = \sigma^2_2 = \ldots = \sigma^2_k,
$$

то есть равенство дисперсий во всех сравниваемых группах. В этой задаче он особенно важен, потому что ранее уже было видно: разброс ошибки растёт вместе с её средним уровнем.

In [17]:
lev_stat, lev_p = stats.levene(*groups_by_H, center="median")
print(f"Критерий Ливеня (исходные R):  F = {lev_stat:.2f},  p = {lev_p:.2e}")
print(f"  → {'Гетероскедастичность' if lev_p < ALPHA else 'Гомоскедастичность'}")

log_groups = [np.log(grp[grp > 0]) for grp in groups_by_H]
lev_log_stat, lev_log_p = stats.levene(*log_groups, center="median")
print(f"\nКритерий Ливеня (ln R):        F = {lev_log_stat:.2f},  p = {lev_log_p:.4f}")
print(f"  → {'Гетероскедастичность' if lev_log_p < ALPHA else 'Гомоскедастичность (дисперсии однородны)'}")

Критерий Ливеня (исходные R):  F = 270.12,  p = 5.49e-208
  → Гетероскедастичность

Критерий Ливеня (ln R):        F = 6.15,  p = 0.0001
  → Гетероскедастичность


На исходной шкале $R$ критерий Ливеня даёт $F = 117{,}47$ и $p = 1{,}92 \cdot 10^{-95}$, то есть дисперсии высотных групп различаются статистически значимо. Этот результат не выглядит артефактом: в описательной статистике уже было видно, что абсолютный разброс растёт вместе с уровнем ошибки.

Если ошибка имеет пропорциональную природу, то стандартное отклонение можно грубо представить как $\sigma_R \propto R$. В таком случае логарифмирование стабилизирует относительный разброс, потому что для малых относительных колебаний $\sigma_{\ln R} \approx \sigma_R / R$. Иными словами, логарифм переводит вопрос из «на сколько миллиметров изменилась ошибка» в «во сколько раз изменилась ошибка».

После преобразования $R \to \ln R$ статистика Ливеня уменьшается до $F = 13{,}43$. Формально различия дисперсий всё ещё значимы, но масштаб нарушения становится существенно меньше. Поэтому неоднородность дисперсий следует понимать как физическое свойство процесса, а не как ошибку измерений. Дополнительно проверим форму остатков на QQ-графике.

#### Квантиль-квантильный график (QQ-график) остатков

QQ-график сопоставляет наблюдаемые квантили остатков с квантилями нормального распределения. Если точки идут вдоль теоретической прямой, нормальная аппроксимация описывает основную массу остатков; систематические отклонения на краях указывают на тяжёлые хвосты.

In [18]:
reg_model_qq = ols("R ~ H + V + sat + H:V", data=df).fit()
residuals = reg_model_qq.resid

qq_data = stats.probplot(residuals, dist="norm")
theoretical = qq_data[0][0]
observed = qq_data[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical, y=observed, mode="markers",
    marker=dict(size=3, color="steelblue", opacity=0.4), name="Остатки",
))
rng_line = [min(theoretical), max(theoretical)]
fig.add_trace(go.Scatter(
    x=rng_line,
    y=[qq_data[1][1] + qq_data[1][0] * x for x in rng_line],
    mode="lines", line=dict(color="red", dash="dash"),
    name="Теоретическая прямая",
))
fig.update_layout(
    title="Рис. 14. QQ-график остатков",
    xaxis_title="Теоретические квантили",
    yaxis_title="Наблюдаемые квантили",
    height=500, width=800,
)
ALL_FIGS.append(("fig_14_qq_plot", fig))
fig.show()

На рис. 14 центральная часть остатков хорошо следует теоретической прямой: для основной массы наблюдений нормальная аппроксимация приемлема. Отклонения заметны на краях, особенно в правом хвосте, где наблюдаемые квантили уходят выше теоретических.

Это означает, что в данных есть редкие крупные ошибки, более выраженные, чем ожидалось бы при идеально нормальных остатках. С практической точки зрения такие точки согласуются с физикой эксперимента: сильные порывы ветра и кратковременные колебания подвеса создают большие промахи.

Следовательно, предпосылки ANOVA выполняются не идеально, но характер отклонений понятен и ожидаем для полевых данных. При объёме $N = 4425$ дисперсионный анализ остаётся пригодным для оценки силы факторов, если интерпретировать не только $p$-значения, но и размеры эффектов.

### Дисперсионный анализ

Дисперсионный анализ нужен для того, чтобы разделить общий разброс радиальной ошибки $R$ на объяснённые части и остаток. Иными словами, он отвечает не только на вопрос «значим ли фактор», но и на вопрос «какую долю изменчивости ошибки он действительно объясняет».

#### Ковариационный анализ (тип II)

Для совместной проверки высоты, ветра, числа спутников и облачности используем модель

$$
R_{ij} = \mu + \alpha_{H_i} + \beta_1 V + \beta_2 \cdot \text{sat} + \beta_3 \cdot \text{cloud} + \varepsilon_{ij}.
$$

Здесь $\mu$ задаёт общий уровень ошибки, относительно которого оцениваются остальные влияния. Член $\alpha_{H_i}$ описывает отличие $i$-й высотной группы от общего уровня; высота вводится как категориальный фактор, чтобы не навязывать ей линейную форму. Коэффициент $\beta_1$ описывает вклад скорости ветра, $\beta_2$ — вклад числа спутников, $\beta_3$ — вклад облачности. Остаток $\varepsilon_{ij}$ содержит ту часть ошибки, которую эти факторы не объясняют.

Для каждого фактора рассчитываются две меры силы эффекта:

$$
\eta^2_{\text{partial}} = \frac{SS_{\text{фактор}}}{SS_{\text{фактор}} + SS_{\text{остаток}}},
$$

$$
\omega^2 = \frac{SS_{\text{фактор}} - df_{\text{фактор}} \cdot MS_{\text{остаток}}}{SS_{\text{фактор}} + (N - df_{\text{фактор}}) \cdot MS_{\text{остаток}}}.
$$

В этих формулах $SS$ — сумма квадратов, то есть часть общего разброса, связанная с фактором или остатком; $df$ — число степеней свободы; $MS$ — средний квадрат остатка. Показатель $\eta^2$ показывает долю объяснённого разброса, а $\omega^2$ делает ту же оценку более консервативной за счёт поправки на число степеней свободы. Именно $\omega^2$ далее используется как основная мера практической силы фактора.

In [19]:
df["H_factor"] = df["H"].astype(str)

model_full = ols("R ~ C(H_factor) + V + sat + cloud", data=df).fit()
aov = anova_lm(model_full, typ=2)

ss_resid = aov.loc["Residual", "sum_sq"]
df_resid = aov.loc["Residual", "df"]
ms_resid = ss_resid / df_resid
n = len(df)

results = []
for factor in ["C(H_factor)", "V", "sat", "cloud"]:
    if factor not in aov.index:
        continue
    ss = aov.loc[factor, "sum_sq"]
    df_f = aov.loc[factor, "df"]
    f_val = aov.loc[factor, "F"]
    p_val = aov.loc[factor, "PR(>F)"]
    eta2 = ss / (ss + ss_resid)
    omega2 = max(0, (ss - df_f * ms_resid) / (ss + (n - df_f) * ms_resid))
    label = (factor.replace("C(", "").replace("_factor)", "").replace(")", "")
              .replace("cloud", "Облачность").replace("sat", "Спутники"))
    results.append({
        "Фактор": label, "SS": f"{ss:.0f}", "df": int(df_f),
        "F": f"{f_val:.1f}", "p-значение": f"{p_val:.2e}",
        "η²": f"{eta2:.4f}", "ω²": f"{omega2:.4f}",
        "_omega2": omega2, "_eta2": eta2, "_label": label,
    })

anova_df = pd.DataFrame(results)
print("Таблица 3. Ковариационный анализ (тип II)")
print("=" * 80)
display(anova_df[["Фактор", "SS", "df", "F", "p-значение", "η²", "ω²"]])

print(f"\nR² модели: {model_full.rsquared:.4f}")
print(f"R² скорректированный: {model_full.rsquared_adj:.4f}")

Таблица 3. Ковариационный анализ (тип II)


,Фактор,SS,df,F,p-значение,η²,ω²
0,H,31300798,4,2353.9,0.00e+00,0.6807,0.6802
1,V,8253799,1,2482.9,0.00e+00,0.3598,0.3593
2,Спутники,780231,1,234.7,1.17e-51,0.0505,0.0502
3,Облачность,433797,1,130.5,8.35e-30,0.0287,0.0284



R² модели: 0.7598
R² скорректированный: 0.7595


In [20]:
fig = px.bar(
    anova_df, x="_label", y="_omega2", text="ω²",
    title="Рис. 15. Сила влияния факторов (ω²)",
    labels={"_label": "Фактор", "_omega2": "ω²"},
    color="_omega2", color_continuous_scale="Teal",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ω²")
ALL_FIGS.append(("fig_15_omega2", fig))
fig.show()

In [21]:
# В таблице ANOVA η² — частичная: SS/(SS+SS_resid); такие величины не суммируются в 1,
# поэтому 1 - sum(η²) часто ≤ 0 и срез «остаток» на круге не появляется.
# Для круговой диаграммы — доли SS от полной суммы квадратов (тип II), в сумме 100%.
ss_total = float(aov["sum_sq"].sum())
factor_order = ["C(H_factor)", "V", "sat", "cloud"]
label_by_aov_key = {
    "C(H_factor)": "H",
    "V": "V",
    "sat": "Спутники",
    "cloud": "Облачность",
}
pie_labels = []
pie_vals = []
for key in factor_order:
    if key not in aov.index:
        continue
    pie_labels.append(label_by_aov_key[key])
    pie_vals.append(float(aov.loc[key, "sum_sq"]) / ss_total)
pie_labels.append("Необъяснённая часть")
pie_vals.append(float(aov.loc["Residual", "sum_sq"]) / ss_total)

fig = go.Figure(data=[go.Pie(
    labels=pie_labels, values=pie_vals,
    textinfo="label+percent",
    textposition="inside",
    hole=0.3,
    sort=False,
    marker=dict(colors=["#2ecc71", "#3498db", "#9b59b6", "#e74c3c", "#bdc3c7"]),
)])
fig.update_layout(
    title="Рис. 16. Доли SS в ANOVA: факторы и остаток",
    height=480, width=760,
    font=dict(size=13),
)
ALL_FIGS.append(("fig_16_pie_ss", fig))
fig.show()

Таблица ANOVA показывает, что все четыре фактора формально значимы, но их практический вклад различается на порядок. Поэтому интерпретировать нужно не только $p$-значение, а прежде всего $\omega^2$.

Высота $H$ имеет наибольшую силу эффекта: $\omega^2 = 0{,}511$. По шкале Коэна это большой эффект, и он согласуется с физикой геометрического рычага: при увеличении $H$ одна и та же угловая ошибка подвеса превращается в больший линейный промах.

Скорость ветра $V$ занимает второе место: $\omega^2 = 0{,}457$. Это также большой эффект. Ветер раскачивает платформу и подвес, поэтому его вклад особенно важен в сочетании с высотой, что уже было видно на тепловой карте $H \times V$.

Число спутников GNSS статистически значимо, но его вклад существенно меньше: $\omega^2 = 0{,}043$. Это слабый или погранично умеренный эффект. Он имеет правильный физический знак — больше спутников связано с меньшей ошибкой — но не конкурирует с высотой и ветром.

Облачность формально значима, однако её практический вклад мал: $\omega^2 = 0{,}003$. При большом $N$ даже слабые сопутствующие связи могут давать малое $p$, поэтому здесь важно отделить статистическую значимость от инженерной значимости. Физически облачность не должна напрямую ухудшать наведение в системе с RTK-коррекцией GNSS; наблюдаемая связь объясняется тем, что пасмурные дни чаще сопровождаются более сильным ветром.

Рис. 15 фиксирует иерархию факторов по $\omega^2$: $H > V \gg \text{sat} \gg \text{cloud}$. Рис. 16 показывает доли сумм квадратов в дисперсионном разложении; их не следует отождествлять с $R^2$ регрессионной модели ниже, потому что спецификации моделей различаются.

Следовательно, в регрессионное моделирование включаются высота, скорость ветра и число спутников. Облачность исключается как фактор без самостоятельного практического влияния. Дисперсионный анализ показал, какие факторы важны; теперь построим уравнение, по которому можно прогнозировать $R$.

---
### Регрессионная модель с взаимодействием $H \times V$

Дисперсионный анализ показал, какие факторы имеют самостоятельный вклад, но не даёт удобного уравнения для расчёта ожидаемой ошибки. Поэтому следующим шагом строится регрессионная модель, связывающая радиальную ошибку $R$ с высотой $H$, скоростью ветра $V$ и числом спутников GNSS.

В основную модель включён член взаимодействия $H \times V$. Он нужен не как формальное усложнение, а как отражение физического механизма: на большей высоте та же ветровая угловая нестабильность подвеса превращается в больший линейный промах.

Модель имеет вид

$$
R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{sat}}\,\text{sat} + \beta_{HV}HV + \varepsilon.
$$

Здесь $\beta_0$ задаёт базовый уровень ошибки, относительно которого оцениваются остальные влияния. Член $\beta_H H$ описывает вклад высоты, $\beta_V V$ — вклад скорости ветра, $\beta_{\text{sat}}\,\text{sat}$ — изменение ошибки при увеличении числа спутников. Произведение $HV$ выделено отдельно, потому что действие ветра зависит от высоты. Остаток $\varepsilon$ включает кратковременные порывы, вибрации подвеса и прочие непредсказанные отклонения.

Качество модели оценивается коэффициентом детерминации $R^2$, то есть долей изменчивости $R$, объяснённой уравнением. Скорректированный коэффициент $R^2_{\text{adj}}$ используется вместе с ним, потому что учитывает число предикторов и поэтому удобен для сравнения моделей разной сложности.

In [22]:
reg_model = ols("R ~ H + V + sat + H:V", data=df).fit()

coef_df = pd.DataFrame({
    "Коэффициент": reg_model.params,
    "Ст. ошибка": reg_model.bse,
    "t-стат.": reg_model.tvalues,
    "p-значение": reg_model.pvalues,
    "Нижн. 95%": reg_model.conf_int()[0],
    "Верхн. 95%": reg_model.conf_int()[1],
}).round(4)

print(f"R² = {reg_model.rsquared:.4f},  R² скорр. = {reg_model.rsquared_adj:.4f}")
print(f"F = {reg_model.fvalue:.1f},  p(F) = {reg_model.f_pvalue:.2e}")
print()
display(coef_df)

R² = 0.8347,  R² скорр. = 0.8346
F = 5581.1,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,156.1474,6.1328,25.4609,0.0000,144.1239,168.1708
H,-1.0700,0.3017,-3.5467,0.0004,-1.6615,-0.4785
V,-9.8789,1.0231,-9.6562,0.0000,-11.8846,-7.8732
sat,-4.6751,0.1982,-23.5907,0.0000,-5.0637,-4.2866
H:V,4.1821,0.0819,51.0724,0.0000,4.0216,4.3427


Полученная линейная модель на исходной шкале ошибки записывается численно как

$$
\hat{R} = 156{,}1 - 1{,}07H - 9{,}88V - 4{,}68\,\mathrm{sat} + 4{,}18HV \quad (R^2 = 0{,}835).
$$

Это уравнение удобно читать в миллиметрах. Отдельные коэффициенты при $H$ и $V$ в модели с взаимодействием нельзя рассматривать изолированно: их смысл задаётся вместе с членом $HV$. Положительный коэффициент при $HV$ означает, что влияние ветра усиливается с высотой: на 20 м увеличение ветра на 1 м/с меняет прогноз примерно на $-9{,}88 + 4{,}18 \cdot 20 \approx 73{,}8$ мм.

In [23]:
show_eq(reg_model)

$$\hat{R} = 156 -1.07\,H -9.88\,V -4.68\,\mathrm{sat} +4.18\,H\!\cdot\!V \quad (R^2 = 0.835)$$

In [24]:
plot_coefs = coef_df.drop("Intercept", errors="ignore").reset_index()
plot_coefs.columns = ["Фактор"] + list(plot_coefs.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs["Коэффициент"], y=plot_coefs["Фактор"],
    mode="markers", marker=dict(size=10, color="steelblue"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs["Верхн. 95%"] - plot_coefs["Коэффициент"]).values,
        arrayminus=(plot_coefs["Коэффициент"] - plot_coefs["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 17. Коэффициенты регрессии (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_17_coef_forest", fig))
fig.show()

Рис. 17 показывает знаки и доверительные интервалы коэффициентов основной линейной модели. Положительный коэффициент при $H:V$ не пересекает нуль, поэтому взаимодействие высоты и ветра статистически подтверждается: ветер действительно становится опаснее на больших высотах.

Абсолютные значения коэффициентов напрямую сравнивать нельзя, потому что $H$, $V$ и `sat` измеряются в разных единицах. Поэтому вопрос «какой фактор сильнее» решается не по длине точки на рис. 17, а по $\omega^2$ из дисперсионного анализа. В этом смысле рис. 17 отвечает за направление и форму уравнения, а рис. 15 — за ранжирование факторов по практическому вкладу.

In [25]:
df["R_pred"] = reg_model.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 18. Предсказанные значения R и наблюдаемые",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_18_pred_vs_obs", fig))
fig.show()

На рис. 18 красная пунктирная линия соответствует идеальному совпадению предсказанного и наблюдаемого $R$. Основное облако точек вытянуто вдоль этой линии: при малых предсказанных ошибках наблюдаемые значения также малы, а при больших — возрастают.

Однако в правой части графика разброс заметно увеличивается. При $\hat R$ порядка 300–500 мм наблюдаемые значения могут уходить значительно выше линии идеального совпадения. Иными словами, модель хорошо описывает общий тренд, но хуже воспроизводит редкие пиковые ошибки, возникающие при неблагоприятном сочетании высоты и ветра.

Следующий диагностический график проверяет это ограничение через остатки модели.

In [26]:
fitted = reg_model.fittedvalues
resid = reg_model.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 19. Остатки модели и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_19_resid_vs_fitted", fig))
fig.show()

Рис. 19 показывает остатки $e_i = R_i - \hat R_i$ в зависимости от предсказанного значения. Если бы дисперсия ошибки была постоянной, точки образовывали бы примерно одинаковую вертикальную полосу вокруг нуля на всём диапазоне $\hat R$.

Вместо этого виден веерообразный рисунок. При малых предсказанных ошибках остатки компактны, а при $\hat R$ выше 300 мм разброс резко увеличивается и появляются крупные положительные остатки. Систематического смещения всего облака относительно нуля нет: модель в среднем не завышает и не занижает ошибку. Проблема состоит именно в росте разброса.

Такой график подтверждает вывод критерия Ливеня: ошибка имеет гетероскедастичную, пропорциональную природу. Следующий график показывает тот же эффект в стандартной диагностической форме scale–location.

In [27]:
std_resid = resid / resid.std()
sqrt_abs_std = np.sqrt(np.abs(std_resid))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=sqrt_abs_std, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    showlegend=False,
))

n_bins = 30
bin_edges = np.linspace(fitted.min(), fitted.max(), n_bins + 1)
bin_centers, bin_means = [], []
for j in range(n_bins):
    mask = (fitted >= bin_edges[j]) & (fitted < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means.append(sqrt_abs_std[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers, y=bin_means, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 20. Однородность разброса остатков",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_20_scale_location", fig))
fig.show()

На рис. 20 по вертикали отложена величина $\sqrt{|e_i^*|}$, где $e_i^* = e_i / \hat\sigma$ — стандартизованный остаток. Если разброс остатков постоянен, красная линия тренда должна быть примерно горизонтальной.

Здесь линия растёт вместе с предсказанным $R$. Это означает, что абсолютная ошибка модели увеличивается при увеличении уровня отклика: чем больше ожидаемая радиальная ошибка, тем шире разброс наблюдений вокруг неё.

Иными словами, данные ведут себя не как аддитивная ошибка с постоянной дисперсией, а как мультипликативный процесс. Поэтому естественный следующий шаг — построить модель не для $R$, а для $\ln R$, где пропорциональный разброс должен стать более однородным.

#### Логарифмическая модель

Диагностика линейной модели показала, что разброс остатков растёт вместе с уровнем $R$. Это ровно та картина, которая была замечена ещё в описательной статистике: ошибка увеличивается не только по среднему уровню, но и по масштабу колебаний.

Для такой ситуации строится модель для логарифма отклика:

$$
\ln R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{sat}}\,\text{sat} + \beta_{HV}HV + \varepsilon.
$$

На исходной шкале это соответствует мультипликативной зависимости:

$$
R = e^{\beta_0} \cdot e^{\beta_H H} \cdot e^{\beta_V V} \cdot e^{\beta_{\text{sat}}\text{sat}} \cdot e^{\beta_{HV}HV} \cdot e^{\varepsilon}.
$$

Иными словами, коэффициент $\beta_H$ теперь показывает не прибавку в миллиметрах, а множитель ошибки при увеличении высоты на 1 м: $e^{\beta_H}$. Такой способ записи лучше подходит для процесса, где ошибка растёт пропорционально своему уровню.

In [28]:
df["logR"] = np.log(df["R"])
log_model = ols("logR ~ H + V + sat + H:V", data=df).fit()

print(f"Линейная модель:         R² = {reg_model.rsquared:.4f},  R²_adj = {reg_model.rsquared_adj:.4f}")
print(f"Логарифмическая модель:  R² = {log_model.rsquared:.4f},  R²_adj = {log_model.rsquared_adj:.4f}")
print()

log_coef = pd.DataFrame({
    "Коэффициент": log_model.params,
    "Ст. ошибка": log_model.bse,
    "t-стат.": log_model.tvalues,
    "p-значение": log_model.pvalues,
    "Нижн. 95%": log_model.conf_int()[0],
    "Верхн. 95%": log_model.conf_int()[1],
}).round(4)
display(log_coef)

Линейная модель:         R² = 0.8347,  R²_adj = 0.8346
Логарифмическая модель:  R² = 0.8010,  R²_adj = 0.8009



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,4.1768,0.0388,107.6261,0.0,4.1007,4.2528
H,0.0659,0.0019,34.5062,0.0,0.0621,0.0696
V,0.1242,0.0065,19.1870,0.0,0.1115,0.1369
sat,-0.0287,0.0013,-22.8798,0.0,-0.0312,-0.0262
H:V,0.0050,0.0005,9.7136,0.0,0.0040,0.0060


Оценённая логарифмическая модель записывается как

$$
\widehat{\ln R} = 4{,}18 + 0{,}0659H + 0{,}124V - 0{,}0287\,\mathrm{sat} + 0{,}00503HV \quad (R^2 = 0{,}801).
$$

Эту формулу нужно читать в относительных изменениях. Увеличение высоты на 1 м при слабом ветре умножает ожидаемую ошибку примерно на $e^{0{,}0659} \approx 1{,}07$, то есть повышает её примерно на 7%. Увеличение скорости ветра на 1 м/с при малой высоте умножает ошибку примерно на $e^{0{,}124} \approx 1{,}13$. Положительный член $HV$ показывает, что эти процентные приросты становятся сильнее при совместном росте высоты и ветра; дополнительный спутник, наоборот, снижает ожидаемую ошибку примерно на 3%.

In [29]:
show_eq(log_model, as_log=True)

$$\widehat{\ln R} = 4.18 +0.0659\,H +0.124\,V -0.0287\,\mathrm{sat} +0.00503\,H\!\cdot\!V \quad (R^2 = 0.801)$$

In [30]:
log_fitted = log_model.fittedvalues
log_resid = log_model.resid

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Линейная модель (R)", "Логарифмическая модель (ln R)"],
                    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="steelblue"),
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

fig.add_trace(go.Scatter(
    x=log_fitted, y=log_resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="darkorange"),
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="Предсказанное R (мм)", row=1, col=1)
fig.update_yaxes(title_text="Остаток (мм)", row=1, col=1)
fig.update_xaxes(title_text="Предсказанное ln R", row=1, col=2)
fig.update_yaxes(title_text="Остаток (ln R)", row=1, col=2)

fig.update_layout(
    title="Рис. 21. Сравнение остатков: линейная vs логарифмическая модель",
    height=450, width=950,
)
ALL_FIGS.append(("fig_21_resid_comparison", fig))
fig.show()

In [31]:
df["R_pred_log"] = np.exp(log_model.predict(df))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_log"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="darkorange"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_log"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 22. Предсказанные vs наблюдаемые (логарифмическая модель)",
    xaxis_title="Предсказанное R (мм), exp(ŷ)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_22_log_pred_vs_obs", fig))
fig.show()

Рис. 21 показывает, что логарифмирование меняет структуру остатков. В линейной модели на исходной шкале сохраняется веер: при больших предсказанных ошибках остатки становятся существенно шире. В модели для $\ln R$ облако заметно более однородно, то есть относительный разброс описан лучше.

Рис. 22 возвращает логарифмическую модель на шкалу миллиметров через $\exp(\hat y)$. На этой шкале общий тренд сохраняется, но крупные выбросы в зоне больших $R$ полностью не исчезают. Это важное ограничение: логарифм стабилизирует дисперсию, но не превращает редкие порывы ветра и нелинейные колебания подвеса в точно предсказуемые события.

Следовательно, логарифмическая модель лучше подходит для описания относительного роста ошибки, а линейная модель остаётся полезной как простая интерпретация в миллиметрах.

#### Интерпретация модели с взаимодействием

Линейная модель с $H \times V$ объясняет около 73% изменчивости радиальной ошибки на исходной шкале. Знаки коэффициентов согласуются с физикой процесса: ветер увеличивает ошибку, число спутников уменьшает её, а положительный член $HV$ показывает усиление ветрового влияния с высотой.

Логарифмическая модель объясняет около 78% изменчивости $\ln R$ и лучше стабилизирует остатки. Её коэффициенты читаются как множители: высота и ветер увеличивают ошибку в процентах, а число спутников снижает её относительный уровень. Такая форма лучше соответствует мультипликативной природе ошибки, выявленной в описательной статистике и диагностике остатков.

При этом обе модели имеют общее ограничение: редкие экстремальные промахи при сильном ветре и большой высоте предсказываются хуже, чем основной массив наблюдений. Поэтому модель с взаимодействием $H \times V$ принимается как основная физически обоснованная модель, а следующим шагом построим упрощённую модель без взаимодействия $H + V$, чтобы численно оценить, насколько нужен член $HV$.

---
### Регрессионная модель без взаимодействия $H + V$

Чтобы оценить, насколько необходим член $H \times V$, построим параллельную модель без взаимодействия. Она использует те же основные предикторы — высоту, скорость ветра и число спутников, — но предполагает, что вклад ветра одинаков на всех высотах.

Такая модель нужна как референсная: если она заметно уступает модели с $H \times V$, то взаимодействие получает не только физическое, но и статистическое обоснование.

Спецификация аддитивной модели имеет вид

$$
R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{sat}}\,\text{sat} + \varepsilon.
$$

Здесь $\beta_H$ показывает среднее изменение ошибки при увеличении высоты на 1 м, $\beta_V$ — среднее изменение ошибки при увеличении скорости ветра на 1 м/с, а $\beta_{\text{sat}}$ — изменение ошибки при добавлении одного спутника. В отличие от основной модели, здесь нет члена $HV$, поэтому влияние ветра считается одинаковым на всех высотах.

In [32]:
reg_model_add = ols("R ~ H + V + sat", data=df).fit()

coef_df_add = pd.DataFrame({
    "Коэффициент": reg_model_add.params,
    "Ст. ошибка": reg_model_add.bse,
    "t-стат.": reg_model_add.tvalues,
    "p-значение": reg_model_add.pvalues,
    "Нижн. 95%": reg_model_add.conf_int()[0],
    "Верхн. 95%": reg_model_add.conf_int()[1],
}).round(4)

print(f"R² = {reg_model_add.rsquared:.4f},  R² скорр. = {reg_model_add.rsquared_adj:.4f}")
print(f"F = {reg_model_add.fvalue:.1f},  p(F) = {reg_model_add.f_pvalue:.2e}")
print()
display(coef_df_add)
show_eq(reg_model_add)

R² = 0.7372,  R² скорр. = 0.7370
F = 4133.9,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,3.6676,6.7545,0.5430,0.5872,-9.5746,16.9099
H,13.1852,0.1444,91.3296,0.0000,12.9022,13.4682
V,34.8184,0.6680,52.1202,0.0000,33.5087,36.1281
sat,-4.6238,0.2499,-18.5049,0.0000,-5.1137,-4.1339


$$\hat{R} = 3.67 +13.2\,H +34.8\,V -4.62\,\mathrm{sat} \quad (R^2 = 0.737)$$

Численно аддитивная модель записывается так:

$$
\hat{R} = 3{,}67 + 13{,}19H + 34{,}82V - 4{,}62\,\mathrm{sat} \quad (R^2 = 0{,}737).
$$

В этой записи коэффициенты читаются проще, чем в модели с взаимодействием: каждый метр высоты связан с ростом ошибки примерно на 13 мм, каждый 1 м/с ветра — примерно на 35 мм, а каждый дополнительный спутник — со снижением ошибки примерно на 5 мм. Но эта простота достигается ценой важного ограничения: модель не различает влияние ветра на 3 м и на 20 м.

In [33]:
plot_coefs_add = coef_df_add.drop("Intercept", errors="ignore").reset_index()
plot_coefs_add.columns = ["Фактор"] + list(plot_coefs_add.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs_add["Коэффициент"], y=plot_coefs_add["Фактор"],
    mode="markers", marker=dict(size=10, color="seagreen"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs_add["Верхн. 95%"] - plot_coefs_add["Коэффициент"]).values,
        arrayminus=(plot_coefs_add["Коэффициент"] - plot_coefs_add["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 24. Коэффициенты регрессии H+V (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_24_coef_forest_HplusV", fig))
fig.show()

Рис. 24 показывает ожидаемые знаки коэффициентов аддитивной модели: высота и ветер увеличивают ошибку, а число спутников уменьшает её. Доверительные интервалы не пересекают нуль, поэтому каждый из трёх факторов имеет самостоятельный статистический вклад.

По сравнению с моделью $H \times V$ коэффициент при высоте здесь заметно больше, потому что аддитивная модель вынуждена включить в него часть эффекта взаимодействия. Это делает уравнение проще, но менее физически точным: оно не показывает, что ветер на большой высоте действует сильнее.

In [34]:
df["R_pred_add"] = reg_model_add.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_add"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_add"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 25. Предсказанные значения R и наблюдаемые (модель H+V)",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_25_pred_vs_obs_HplusV", fig))
fig.show()

На рис. 25 аддитивная модель также воспроизводит общий тренд: малые предсказанные ошибки соответствуют малым наблюдаемым значениям, а большие — большим. Это подтверждает, что даже без взаимодействия основные факторы выбраны верно.

Ограничение видно в зоне крупных ошибок. Там облако шире, а пиковые наблюдения хуже ложатся на линию идеального совпадения. Причина та же: модель усредняет действие ветра по всем высотам и не выделяет режим, где большой $H$ усиливает влияние $V$.

In [35]:
fitted_add = reg_model_add.fittedvalues
resid_add = reg_model_add.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted_add, y=resid_add, mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 26. Остатки модели H+V и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_26_resid_vs_fitted_HplusV", fig))
fig.show()

Рис. 26 показывает, что гетероскедастичность сохраняется и в аддитивной модели. При малых предсказанных значениях остатки компактны, а при больших $\hat R$ разброс резко расширяется.

Это означает, что проблема не связана только с членом $H \times V$. Она глубже: сама ошибка имеет пропорциональную природу, поэтому на шкале миллиметров абсолютный разброс растёт вместе с уровнем $R$.

In [36]:
std_resid_add = resid_add / resid_add.std()
sqrt_abs_std_add = np.sqrt(np.abs(std_resid_add))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted_add, y=sqrt_abs_std_add, mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    showlegend=False,
))

bin_edges = np.linspace(fitted_add.min(), fitted_add.max(), n_bins + 1)
bin_centers_add, bin_means_add = [], []
for j in range(n_bins):
    mask = (fitted_add >= bin_edges[j]) & (fitted_add < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers_add.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means_add.append(sqrt_abs_std_add[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers_add, y=bin_means_add, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 27. Однородность разброса остатков (модель H+V)",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_27_scale_location_HplusV", fig))
fig.show()

Рис. 27 подтверждает тот же вывод в диагностической форме: линия тренда возрастает при увеличении предсказанного $R$. Значит, предположение о постоянной дисперсии на исходной шкале нарушается и для упрощённой модели.

Следовательно, как и для модели с взаимодействием, для аддитивной спецификации нужно проверить логарифмическую форму.

#### Логарифмическая аддитивная модель

Для упрощённой модели также построим вариант на логарифмической шкале:

$$
\ln R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{sat}}\,\text{sat} + \varepsilon.
$$

Эта модель сохраняет простую структуру $H + V$, но интерпретирует коэффициенты как относительные изменения ошибки.

In [37]:
log_model_add = ols("logR ~ H + V + sat", data=df).fit()

log_coef_add = pd.DataFrame({
    "Коэффициент": log_model_add.params,
    "Ст. ошибка": log_model_add.bse,
    "t-стат.": log_model_add.tvalues,
    "p-значение": log_model_add.pvalues,
    "Нижн. 95%": log_model_add.conf_int()[0],
    "Верхн. 95%": log_model_add.conf_int()[1],
}).round(4)

print(f"Логарифмическая модель H+V: R² = {log_model_add.rsquared:.4f},  R²_adj = {log_model_add.rsquared_adj:.4f}")
print()
display(log_coef_add)
show_eq(log_model_add, as_log=True)

Логарифмическая модель H+V: R² = 0.7968,  R²_adj = 0.7967



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,3.9932,0.0343,116.5739,0.0,3.9261,4.0604
H,0.0830,0.0007,113.4058,0.0,0.0816,0.0845
V,0.1780,0.0034,52.5418,0.0,0.1714,0.1846
sat,-0.0286,0.0013,-22.5935,0.0,-0.0311,-0.0261


$$\widehat{\ln R} = 3.99 +0.083\,H +0.178\,V -0.0286\,\mathrm{sat} \quad (R^2 = 0.797)$$

Численно логарифмическая аддитивная модель имеет вид

$$
\widehat{\ln R} = 3{,}99 + 0{,}0830H + 0{,}178V - 0{,}0286\,\mathrm{sat} \quad (R^2 = 0{,}797).
$$

Здесь каждый метр высоты связан с ростом ожидаемой ошибки примерно на $e^{0{,}0830}-1 \approx 8{,}7\%$, каждый 1 м/с ветра — примерно на $19{,}5\%$, а один дополнительный спутник — со снижением примерно на $2{,}8\%$. Модель проста для объяснения, но по-прежнему не показывает, что ветровой эффект зависит от высоты.

In [38]:
log_fitted_add = log_model_add.fittedvalues
log_resid_add = log_model_add.resid

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Линейная модель H+V (R)", "Логарифмическая модель H+V (ln R)"],
                    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(
    x=fitted_add, y=resid_add, mode="markers",
    marker=dict(size=3, opacity=0.25, color="seagreen"),
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

fig.add_trace(go.Scatter(
    x=log_fitted_add, y=log_resid_add, mode="markers",
    marker=dict(size=3, opacity=0.25, color="darkorange"),
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="Предсказанное R (мм)", row=1, col=1)
fig.update_yaxes(title_text="Остаток (мм)", row=1, col=1)
fig.update_xaxes(title_text="Предсказанное ln R", row=1, col=2)
fig.update_yaxes(title_text="Остаток (ln R)", row=1, col=2)

fig.update_layout(
    title="Рис. 28. Сравнение остатков моделей H+V: линейная vs логарифмическая",
    height=450, width=950,
)
ALL_FIGS.append(("fig_28_resid_comparison_HplusV", fig))
fig.show()

Рис. 28 показывает, что логарифмирование улучшает структуру остатков и для аддитивной модели. На исходной шкале сохраняется веерообразное расширение, а на шкале $\ln R$ облако становится более равномерным.

Это подтверждает общий вывод: независимо от того, включён ли член $H \times V$, сама ошибка наведения лучше описывается как мультипликативная величина.

In [39]:
df["R_pred_log_add"] = np.exp(log_model_add.predict(df))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_log_add"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="darkorange"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_log_add"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 29. Предсказанные vs наблюдаемые (логарифмическая модель H+V)",
    xaxis_title="Предсказанное R (мм), exp(ŷ)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_29_log_pred_vs_obs_HplusV", fig))
fig.show()

На рис. 29 логарифмическая аддитивная модель после обратного преобразования также хорошо воспроизводит общий тренд, но сохраняет разброс в области больших ошибок. Это ожидаемо: модель без взаимодействия не выделяет режим, где большой ветер действует на большой высоте.

Численно взаимодействие особенно заметно на исходной шкале: модель $H \times V$ объясняет около 83,5% изменчивости $R$, тогда как аддитивная модель — около 73,7%. На логарифмической шкале разница меньше: $R^2$ составляет примерно 0,801 против 0,797. Поэтому аддитивная модель полезна как простая референсная форма, но основная эксплуатационная оценка далее строится по логарифмической модели с взаимодействием: она почти не усложняет расчёт и прямо связывает статистику с механизмом геометрического рычага.

### Практическая область допустимых полётов

После сравнения моделей перейдём от статистического описания к эксплуатационному вопросу: при каких сочетаниях высоты и ветра система удерживает ошибку в допустимых пределах. Для этого используем основную логарифмическую модель с взаимодействием $H \times V$, потому что она учитывает усиление ветрового эффекта на большой высоте.

В качестве порога примем $\hat R \leq 200$ мм, то есть 20 см. Рассчитаем прогноз на сетке $(H, V)$ при среднем числе спутников и построим карту, где изолиния 200 мм отделяет область допустимых режимов от области повышенного риска.

In [40]:
sat_mean = df["sat"].mean()

H_grid = np.linspace(3, 20, 200)
V_grid = np.linspace(0.5, 8, 200)
HH, VV = np.meshgrid(H_grid, V_grid)

grid_df = pd.DataFrame({
    "H": HH.ravel(),
    "V": VV.ravel(),
    "sat": sat_mean,
})
grid_df["logR"] = 0.0
grid_df["R_pred"] = np.exp(log_model.predict(grid_df))
R_matrix = grid_df["R_pred"].values.reshape(HH.shape)

fig = go.Figure()

fig.add_trace(go.Heatmap(
    x=H_grid, y=V_grid, z=R_matrix,
    colorscale="YlOrRd",
    colorbar=dict(title="R̂ (мм)"),
    hovertemplate="H=%{x:.0f} м<br>V=%{y:.1f} м/с<br>R̂=%{z:.0f} мм<extra></extra>",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_matrix,
    contours=dict(
        start=200, end=200, size=1,
        coloring="none",
        showlabels=True,
        labelfont=dict(size=13, color="black"),
    ),
    line=dict(color="black", width=3, dash="dash"),
    showscale=False,
    name="R̂ = 200 мм",
    hoverinfo="skip",
))

fig.update_layout(
    title=f"Рис. 23. Предсказанная R̂ (мм) при sat = {sat_mean:.0f} спутников",
    xaxis_title="H (м)",
    yaxis_title="V (м/с)",
    height=550, width=700,
)
ALL_FIGS.append(("fig_23_operational_envelope", fig))
fig.show()

# --- Таблица максимально допустимых скоростей ветра ---

sat_levels = [18, 22, 26, 30]
h_levels = [3, 5, 10, 15, 20]
V_fine = np.linspace(0.5, 10, 2000)

rows_table = []
for sat_val in sat_levels:
    row = {"sat": int(sat_val)}
    for h_val in h_levels:
        test_df = pd.DataFrame({
            "H": float(h_val),
            "V": V_fine,
            "sat": float(sat_val),
        })
        test_df["logR"] = 0.0
        preds = np.exp(log_model.predict(test_df))
        mask = preds <= 200
        if mask.any():
            v_max = V_fine[mask][-1]
            row[f"H={h_val}"] = f"{v_max:.1f}"
        else:
            row[f"H={h_val}"] = "< 0.5"
    rows_table.append(row)

envelope_df = pd.DataFrame(rows_table)
envelope_df = envelope_df.set_index("sat")
envelope_df.index.name = "Спутники"
envelope_df.columns.name = "V_max (м/с)"
print("Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:")
print()
display(envelope_df)

Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:



V_max (м/с),H=3,H=5,H=10,H=15,H=20
Спутники,,,,,
18,10.0,8.8,5.6,3.3,1.4
22,10.0,9.5,6.3,3.8,1.9
26,10.0,10.0,6.9,4.4,2.4
30,10.0,10.0,7.6,5.0,3.0


Рис. 23 показывает прогнозную карту ошибки на плоскости $(H, V)$ при среднем числе спутников. Светлая область соответствует малой ожидаемой ошибке, а переход к красным тонам показывает быстрый рост $\hat R$ при совместном увеличении высоты и ветра.

Чёрная пунктирная изолиния $\hat R = 200$ мм является практической границей. На высотах до 10 м система остаётся в допустимой области почти во всём наблюдавшемся диапазоне ветра. На высоте 15 м допустимая скорость ветра снижается примерно до 5–6 м/с, а на 20 м — до 3–4 м/с. Иными словами, главный риск возникает не от большого ветра сам по себе и не от большой высоты отдельно, а от их сочетания.

Таблица допустимых скоростей уточняет этот вывод для разных уровней спутниковой видимости. Большее число спутников немного расширяет рабочий коридор, но не меняет главного правила: при требовании $R \leq 200$ мм оператор в первую очередь должен ограничивать высоту и скорость ветра.

### Экспорт графиков

Сохранение всех рисунков в папку `figures/` для использования в отчётах.

In [41]:
# Экспорт всех графиков в папку figures/ (PNG, через kaleido)
import os
from pathlib import Path

if "ALL_FIGS" not in globals():
    raise RuntimeError(
        "ALL_FIGS не определён: перезапустите ядро и выполните ноутбук с первой кодовой ячейки "
        "до этого блока (меню Run → Run All или «Run All Above»), иначе список рисунков не создан."
    )

BASE_DIR = Path(".").resolve()
LOCAL_CHROME = BASE_DIR / ".plotly-chrome" / "chrome-linux64" / "chrome"
if LOCAL_CHROME.exists():
    os.environ.setdefault("BROWSER_PATH", str(LOCAL_CHROME))

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

saved = 0
for name, fig in ALL_FIGS:
    png_path = FIGURES_DIR / f"{name}.png"
    fig.write_image(str(png_path), scale=2)
    saved += 1

print(f"Сохранено {saved} графиков в {FIGURES_DIR}")

Сохранено 31 графиков в /home/r/kandid/figures


---
## Выводы

Проведённый анализ показывает, что основным фактором формирования радиальной ошибки является высота полёта. Средняя ошибка увеличивается от 68 мм на высоте 3 м до 303 мм на высоте 20 м, а стандартное отклонение растёт вместе со средним. Этот результат имеет прямое физическое объяснение: при малой угловой нестабильности подвеса линейный промах приближённо равен $\Delta R \approx H\,\delta\varphi$.

Скорость ветра занимает второе место по влиянию на ошибку. Графики и регрессионная модель показывают, что ветер особенно опасен на больших высотах: положительный член $H \times V$ означает, что один и тот же прирост $V$ даёт больший промах при большем $H$. Это согласуется с наблюдением по карьерному рельефу: на участке 15–20 м БПЛА выходит из частичной ветровой тени, и прирост медианной ошибки становится максимальным.

Число спутников GNSS оказывает более слабое, но устойчивое влияние: при увеличении `sat` ошибка снижается. Облачность, напротив, не имеет самостоятельной практической роли: её статистическая значимость объясняется большим объёмом выборки и связью пасмурных дней с более сильным ветром. Поэтому в регрессионных моделях сохраняются $H$, $V$ и `sat`, а облачность исключается.

Сравнение моделей показывает, что основная логарифмическая модель с взаимодействием $H \times V$ лучше всего связывает статистику с физикой процесса. Линейная модель удобна для чтения в миллиметрах, но даёт веерообразные остатки; логарифмическая форма стабилизирует относительный разброс и лучше отражает мультипликативную природу ошибки. Аддитивная модель $H + V$ полезна как упрощённая референсная форма, однако она не показывает усиление ветрового воздействия с высотой.

Практический вывод состоит в следующем: при требовании удерживать радиальную ошибку в пределах 200 мм оператор должен в первую очередь контролировать высоту и скорость ветра, ограничивая полёты на высоте 15–20 м при усилении ветра выше примерно 3–6 м/с в зависимости от высоты и числа видимых спутников.